In [118]:
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field, field_validator
from langchain_core.messages import HumanMessage, SystemMessage
from uuid import UUID
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
import operator
from langgraph.types import Send
import re
from pathlib import Path

In [119]:
load_dotenv()

True

In [120]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    max_retries=5,
)


In [121]:
# ============================================================
# 1. Writing configuration
# ============================================================


class BlogConfig(BaseModel):
    tone: Literal[
        "professional",
        "conversational",
        "educational",
        "persuasive",
        "analytical",
    ] = "professional"

    audience: str = "General readers interested in the topic"

    reading_level: Literal["beginner", "intermediate", "advanced"] = "intermediate"

    point_of_view: Literal["first_person", "second_person", "third_person"] = (
        "second_person"
    )

    style_instructions: list[str] = Field(
        default_factory=lambda: [
            "Write naturally and clearly",
            "Use specific explanations rather than vague claims",
            "Vary sentence length naturally",
            "Use active voice where appropriate",
            "Avoid repetitive introductions and conclusions",
            "Avoid exaggerated marketing language",
        ]
    )

    heading_style: Literal["descriptive", "question_based", "benefit_oriented"] = (
        "descriptive"
    )

    use_bullet_points: bool = True
    use_numbered_lists: bool = True
    include_examples: bool = True
    include_faq: bool = False

    seo_priority: Literal["low", "balanced", "high"] = "balanced"

    target_words: int = Field(default=1200, ge=300, le=5000)

    avoid_phrases: list[str] = Field(
        default_factory=lambda: [
            "In today's fast-paced world",
            "Let's dive in",
            "In conclusion, it is important to note",
        ]
    )


# ============================================================
# 2. Planning schema
# ============================================================


class BlogSectionTask(BaseModel):
    id: str = Field(description="Unique section ID, e.g. section_1")

    heading: str = Field(description="Descriptive H2 heading for the section")

    purpose: str = Field(description="What the reader should understand after reading")

    key_points: list[str] = Field(
        min_length=2, description="Specific points the section must explain"
    )

    subtopics: list[str] = Field(
        min_length=2,
        max_length=5,
        description=(
            "Two to five specific H3 headings. "
            "Each heading must cover a distinct aspect of the section."
        ),
    )

    # Allow smaller sections where appropriate.
    suggested_words: int = Field(
        default=250,
        ge=80,
        le=1000,
        description="Approximate word target based on topic complexity",
    )

    include_example: bool = False

    @field_validator("id", mode="before")
    @classmethod
    def normalize_id(cls, value):
        if isinstance(value, int):
            return f"section_{value}"
        return value


class BlogPlan(BaseModel):
    blog_title: str
    meta_description: str
    search_intent: str
    primary_keyword: str
    related_keywords: list[str]
    introduction_brief: str
    sections: list[BlogSectionTask] = Field(min_length=3, max_length=10)
    conclusion_brief: str


# ============================================================
# 3. Section output schema
# ============================================================


from pydantic import BaseModel, Field


class BlogSectionTask(BaseModel):
    id: str
    heading: str
    purpose: str
    key_points: list[str]

    # The planner must decide what each subsection will explain.
    subtopics: list[str] = Field(
        min_length=2,
        max_length=5,
        description=(
            "Specific H3 subsections required in this section. "
            "Each must address a distinct aspect of the topic."
        ),
    )

    suggested_words: int = Field(default=350, ge=150, le=1000)
    include_example: bool = False


class SubsectionDraft(BaseModel):
    heading: str = Field(
        description="A descriptive H3 heading, without Markdown symbols"
    )
    explanation: str = Field(
        description=(
            "Detailed explanation covering what the concept is, "
            "how or why it works, and its practical significance."
        )
    )
    example: str = Field(
        default="",
        description=(
            "A concrete example, code example, or scenario where useful. "
            "Empty if an example would not add value."
        ),
    )


class SectionDraft(BaseModel):
    task_id: str
    heading: str

    opening: str = Field(description="An introductory paragraph for this section")

    subsections: list[SubsectionDraft] = Field(
        min_length=2,
        max_length=5,
    )

    closing_insight: str = Field(
        description="A useful takeaway or transition from the section"
    )

    key_points_covered: list[str]


# ============================================================
# 4. Quality report schema
# ============================================================


class QualityReport(BaseModel):
    relevance_score: int = Field(ge=1, le=5)
    readability_score: int = Field(ge=1, le=5)
    structure_score: int = Field(ge=1, le=5)
    repetition_detected: bool
    unsupported_claims_detected: bool
    keyword_stuffing_detected: bool
    issues: list[str]
    revision_instructions: list[str]
    approved: bool


# ============================================================
# 5. Graph state
# ============================================================


class State(TypedDict, total=False):
    topic: str
    config: BlogConfig
    plan: BlogPlan
    sections: Annotated[list[SectionDraft], operator.add]
    final: str
    output_path: str


In [122]:
# JSON mode is used because the previous Groq error came from
# response_format=json_object without the word "json" in a message.
planner_llm = llm.with_structured_output(
    BlogPlan,
    method="json_mode",
)

writer_llm = llm.with_structured_output(
    SectionDraft,
    method="json_mode",
)


In [123]:
# ============================================================
# 7. Shared writing instructions
# ============================================================


def build_writing_system_prompt(config: BlogConfig) -> str:
    avoid_phrases = "\n".join(f"- {phrase}" for phrase in config.avoid_phrases)
    style_instructions = "\n".join(f"- {item}" for item in config.style_instructions)

    return f"""
You are a professional blog writer, content strategist, and editor.

WRITING PROFILE
- Tone: {config.tone}
- Audience: {config.audience}
- Reading level: {config.reading_level}
- Point of view: {config.point_of_view}
- Heading style: {config.heading_style}
- SEO priority: {config.seo_priority}
- Target article length: approximately {config.target_words} words

VOICE AND READABILITY
- Write naturally, clearly, and professionally.
- Prefer concrete explanations over vague claims.
- Vary sentence and paragraph lengths naturally.
- Explain technical terms when the audience needs it.
- Avoid filler, repetition, and exaggerated claims.
- Never invent personal experiences, statistics, quotations, or sources.
- Never claim that content is 100% human-written.

FORMATTING
- Use Markdown headings with a logical hierarchy.
- Use bullet points: {config.use_bullet_points}
- Use numbered lists: {config.use_numbered_lists}
- Include examples when appropriate: {config.include_examples}
- Include an FAQ section: {config.include_faq}
- Use paragraphs for explanation and analysis.
- Avoid excessive bolding and unnecessary headings.

SEO
- Match the user's search intent.
- Use relevant keywords naturally.
- Avoid keyword stuffing.
- Prioritize usefulness and accuracy over keyword frequency.

PHRASES TO AVOID
{avoid_phrases}

ADDITIONAL STYLE INSTRUCTIONS
{style_instructions}

FACTUAL RELIABILITY
- Do not fabricate facts, studies, citations, or URLs.
- Distinguish uncertainty from established facts.
- Do not claim that information has been verified when it has not.

JSON OUTPUT REQUIREMENT
When asked for structured output, return valid JSON as required by the
provided schema. The word JSON must appear in these instructions.
Do not return Markdown fences or commentary around the JSON.
Follow the structured-output schema exactly.
"""


In [124]:


# ============================================================
# 8. Orchestrator
# ============================================================


def orchestrator(state: State) -> dict:
    config = state["config"]
    topic = state["topic"]

    system_prompt = (
        build_writing_system_prompt(config)
        + +"""
You are a senior content strategist and professional blog architect.

Your task is to create a detailed, logically structured blog plan.
Do not write the finished article.

## 1. ARTICLE PLANNING

Create a plan that:
- Answers the reader's actual search intent.
- Progresses logically from foundational concepts to deeper explanations.
- Uses a specific, accurate title and concise meta description.
- Identifies one primary keyword and relevant related keywords.
- Defines a clear introduction brief and conclusion brief.
- Contains 3–10 distinct sections.
- Avoids overlapping sections and repeated explanations.
- Allocates word counts realistically to approach the configured article length.
- Includes an FAQ section when the writing configuration requests one.

## 2. REQUIRED SECTION STRUCTURE

Every item in the sections array MUST contain all seven fields:

1. id
2. heading
3. purpose
4. key_points
5. subtopics
6. suggested_words
7. include_example

Do not omit any field or return null for a required field.

### id
Use a unique string identifier such as "section_1", "section_2", etc.

### heading
Provide a specific, descriptive H2 heading that communicates the section's purpose.

### purpose
Explain what the reader should understand after reading this section.

### key_points
Provide at least two specific concepts, facts, mechanisms, or practical questions
that the section must explain.

### subtopics
This field is REQUIRED and must contain 2–5 descriptive H3 headings.

Each subtopic must:
- Address a distinct concept or specific question.
- Contribute meaningful information to the section's purpose.
- Be specific enough to guide detailed writing.
- Avoid generic headings such as "Overview" or "Benefits" unless appropriate.

Return the exact H3 heading text in this array.

### suggested_words
Provide an integer between 80 and 1000.

Choose the word target according to the complexity of the section.
Use approximately 150–350 words for ordinary sections and more for complex
sections when necessary. Shorter sections may use 80–149 words when justified.

Allocate enough words to explain the assigned subtopics meaningfully.

### include_example
Return true when a practical example, scenario, or code example would help.
Otherwise, return false.

## 3. CONTENT DEPTH AND QUALITY

Plan sections that enable the writer to explain, where relevant:
- What a concept is.
- How or why it works.
- Why it matters.
- How it is applied in practice.
- Relevant trade-offs, limitations, or alternatives.

Do not force every dimension into every section.

Avoid vague plans that merely instruct the writer to "discuss" a topic.
Specify what needs to be explained and what the reader should learn.

## 4. WORD-COUNT ALLOCATION

Use the configured target_words as the approximate total article length.
Account for the introduction, all sections, and conclusion.

Do not allocate the entire target word count to each section.
Do not create excessive sections with unrealistically small word targets.
Prefer fewer, well-developed sections over many shallow sections.

## 5. SEO AND ACCURACY

- Match the likely search intent.
- Select relevant keywords naturally.
- Avoid keyword stuffing and repetitive headings.
- Do not invent facts, statistics, studies, or sources.
- Do not claim that information has been verified when it has not.

## 6. REQUIRED JSON OUTPUT

Return valid JSON matching the BlogPlan schema exactly.
The word JSON must appear in these instructions.

The top-level JSON object MUST contain:
- blog_title
- meta_description
- search_intent
- primary_keyword
- related_keywords
- introduction_brief
- sections
- conclusion_brief

Every object inside sections MUST contain:
- id
- heading
- purpose
- key_points
- subtopics
- suggested_words
- include_example

Return JSON only. Do not include Markdown code fences, explanations, or commentary.
"""
    )

    result = planner_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(
            content=(
                f"Create a complete blog plan for this topic:\n{topic}\n\n"
                f"Writing configuration in JSON:\n"
                f"{config.model_dump_json(indent=2)}"
            )
        ),
    ])

    # Extra validation beyond the structured-output call.
    if not isinstance(result, BlogPlan):
        result = BlogPlan.model_validate(result)

    ids = [task.id for task in result.sections]
    if len(ids) != len(set(ids)):
        raise ValueError("Planner generated duplicate section IDs.")

    headings = [task.heading.strip().casefold() for task in result.sections]
    if len(headings) != len(set(headings)):
        raise ValueError("Planner generated duplicate section headings.")

    return {
        "plan": result,
        "sections": [],
    }



In [125]:
# ============================================================
# 9. Fan out section tasks in parallel
# ============================================================


def fanout(state: State):
    plan = state["plan"]

    return [
        Send(
            "worker",
            {
                "topic": state["topic"],
                "config": state["config"],
                "plan": plan,
                "task": task,
            },
        )
        for task in plan.sections
    ]


In [126]:

# ============================================================
# 10. Worker
# ============================================================

def worker(payload: dict) -> dict:
    config: BlogConfig = payload["config"]
    plan: BlogPlan = payload["plan"]
    task: BlogSectionTask = payload["task"]

    system_prompt = f"""
You are an experienced subject-matter writer, professional blog writer,
and technical editor.

Write one high-quality, detailed section of a larger blog article.

WRITING PROFILE
- Audience: {config.audience}
- Tone: {config.tone}
- Reading level: {config.reading_level}
- Point of view: {config.point_of_view}
- SEO priority: {config.seo_priority}
- Use examples when appropriate: {config.include_examples}
- Use bullet points: {config.use_bullet_points}
- Use numbered lists: {config.use_numbered_lists}

CONTENT DEPTH
Do not merely list, mention, or briefly define concepts.
Develop each important idea sufficiently for the intended reader to
understand it.

Where relevant, explain:
- What the concept means.
- How or why it works.
- Why it matters in practice.
- How it can be applied.
- Relevant benefits, limitations, trade-offs, or alternatives.

Do not force irrelevant explanations into a section.
Avoid shallow statements, unexplained terminology, and repetitive padding.

REQUIRED STRUCTURE
1. Write an informative opening paragraph that introduces this section.
2. Write a separate H3 subsection for EVERY assigned subtopic.
3. Use the exact assigned subtopic headings.
4. Develop each subsection with multiple meaningful paragraphs when
   the topic requires them.
5. Use bullets for grouped information and numbered lists for sequences.
6. Include a relevant example, scenario, or code snippet when requested
   and appropriate.
7. Explain examples rather than merely inserting them.
8. End with a useful insight or transition, not a generic summary.

EXPLANATION QUALITY
- Explain mechanisms and reasoning, not just outcomes.
- Prefer concrete details over vague generalizations.
- Use examples that directly illustrate the concepts being explained.
- For technical subjects, include code only when it genuinely helps.
- Explain important code rather than dumping unexplained snippets.
- Do not replace detailed prose with a list of buzzwords.

NATURAL WRITING
- Write clearly, professionally, and naturally.
- Vary sentence and paragraph lengths.
- Avoid filler, repetition, clichés, exaggerated claims, and generic AI prose.
- Avoid repeating material assigned to other sections.
- Do not invent statistics, studies, quotations, citations, or sources.

Avoid these phrases:
{json.dumps(config.avoid_phrases, ensure_ascii=False)}

SEO
Use the primary keyword and related keywords naturally.
Do not force keywords into every heading or paragraph.
Prioritize reader usefulness, accuracy, and clarity.

LENGTH
Aim for approximately {task.suggested_words} words.
Give each assigned subtopic enough space for a meaningful explanation.
Do not sacrifice depth just to meet a word count.
Do not inflate the article with repeated points.

CONSISTENCY
- Use the exact assigned section heading.
- Use the exact task ID.
- Cover all assigned key points.
- Follow the assigned purpose and subtopic outline.
- Write only this section, not the entire article.
- Do not write the article's introduction or conclusion unless assigned.

OUTPUT FORMAT
Return valid JSON matching the SectionDraft schema exactly.

Required fields:
- task_id
- heading
- opening
- subsections
- closing_insight
- key_points_covered

Each subsection must contain:
- heading
- explanation
- example

The example field may be an empty string when an example is unnecessary.
Return JSON only, without Markdown code fences or commentary.
"""

    # Send only the context this worker needs.
    task_input = {
        "topic": payload["topic"],
        "blog_title": plan.blog_title,
        "primary_keyword": plan.primary_keyword,
        "related_keywords": plan.related_keywords,
        "section_task": task.model_dump(),
    }

    result = writer_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(
            content=(
                "Write the assigned blog section using this input JSON:\n"
                + json.dumps(task_input, ensure_ascii=False)
            ),
        ),
    ])

    if not isinstance(result, SectionDraft):
        result = SectionDraft.model_validate(result)

    # Validate task identity.
    if result.task_id != task.id:
        raise ValueError(
            f"Expected task ID {task.id!r}, "
            f"received {result.task_id!r}"
        )

    # Validate the main heading.
    if result.heading.strip() != task.heading.strip():
        raise ValueError(
            f"Expected heading {task.heading!r}, "
            f"received {result.heading!r}"
        )

    # Validate required subtopics.
    expected_subtopics = {
        heading.strip().casefold()
        for heading in task.subtopics
    }

    actual_subtopics = {
        subsection.heading.strip().casefold()
        for subsection in result.subsections
    }

    missing_subtopics = expected_subtopics - actual_subtopics

    if missing_subtopics:
        raise ValueError(
            f"Section {task.id!r} omitted required subtopics: "
            f"{sorted(missing_subtopics)}"
        )

    # Reject empty or obviously incomplete output.
    if not result.opening.strip():
        raise ValueError(
            f"Section {task.id!r} has an empty opening."
        )

    for subsection in result.subsections:
        if not subsection.explanation.strip():
            raise ValueError(
                f"Subsection {subsection.heading!r} has no explanation."
            )

    if not result.closing_insight.strip():
        raise ValueError(
            f"Section {task.id!r} has an empty closing insight."
        )

    return {"sections": [result]}


In [127]:
# ============================================================
# 11. Final assembly
# ============================================================


def reducer(state: State) -> dict:
    plan: BlogPlan = state["plan"]
    config: BlogConfig = state["config"]
    sections: list[SectionDraft] = state.get("sections", [])

    expected_ids = [task.id for task in plan.sections]
    section_by_id = {}

    for section in sections:
        if section.task_id in section_by_id:
            raise ValueError(f"Duplicate worker output for {section.task_id!r}.")
        section_by_id[section.task_id] = section

    missing_ids = [task_id for task_id in expected_ids if task_id not in section_by_id]
    unexpected_ids = [
        task_id for task_id in section_by_id if task_id not in expected_ids
    ]

    if missing_ids or unexpected_ids:
        raise ValueError(
            f"Section output mismatch. Missing: {missing_ids}; "
            f"unexpected: {unexpected_ids}"
        )

    # Preserve the planner's original section order, not the order
    # in which parallel workers happened to finish.
    ordered_sections = [section_by_id[task_id] for task_id in expected_ids]

    body_parts = [
        plan.introduction_brief.strip(),
    ]

    for section in ordered_sections:
        body_parts.append(section.body_markdown.strip())

    body_parts.append(plan.conclusion_brief.strip())

    final_md = (
        f"# {plan.blog_title}\n\n"
        f"> {plan.meta_description}\n\n"
        + "\n\n".join(part for part in body_parts if part)
        + "\n"
    )

    # These briefs are instructions, not finished prose. This is
    # intentionally corrected in the next section below.
    # See the note following the code.

    safe_name = re.sub(
        r'[<>:"/\\|?*\x00-\x1f]',
        "",
        plan.blog_title.lower(),
    )
    safe_name = re.sub(r"\s+", "_", safe_name).strip("._")
    filename = (safe_name or "blog")[:100] + ".md"

    output_dir = Path("generated_blogs")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename
    output_path.write_text(final_md, encoding="utf-8")

    print(f"Blog saved to: {output_path.resolve()}")

    return {
        "final": final_md,
        "output_path": str(output_path.resolve()),
    }


In [128]:
# ============================================================
# 12. Build and compile the graph
# ============================================================

graph = StateGraph(State)

graph.add_node("orchestrator", orchestrator)
graph.add_node("worker", worker)
graph.add_node("reducer", reducer)

graph.add_edge(START, "orchestrator")
graph.add_conditional_edges("orchestrator", fanout, ["worker"])
graph.add_edge("worker", "reducer")
graph.add_edge("reducer", END)

app = graph.compile()


In [129]:
# ============================================================
# 13. Run the graph
# ============================================================

if __name__ == "__main__":
    config = BlogConfig(
        tone="professional",
        audience="General readers interested in the topic",
        reading_level="intermediate",
        point_of_view="second_person",
        heading_style="descriptive",
        use_bullet_points=True,
        use_numbered_lists=True,
        include_examples=True,
        include_faq=False,
        seo_priority="balanced",
        target_words=1200,
    )

    output = app.invoke(
        {
            "topic": "What is JEV?",
            "config": config,
            "sections": [],
        },
        config={
            "max_concurrency": 2,
        },
    )

    print("\nBlog generation completed.")
    print(output["final"])
    print(f"\nSaved to: {output['output_path']}")


TypeError: bad operand type for unary +: 'str'